# All-Pairs Shortest Path (APSP) and the Floyd-Warshall Algorithm

This lecture introduces the **All-Pairs Shortest Path (APSP)** problem and details the **Floyd-Warshall algorithm**, a dynamic programming approach used to solve it efficiently.

## 1. Introduction to All-Pairs Shortest Path (APSP)
The APSP problem aims to find the shortest path between **every pair of vertices** $(u, v)$ in a directed, weighted graph.

### Comparison with Single-Source Shortest Path (SSSP)
While SSSP (like Dijkstra or Bellman-Ford) finds the shortest path from one specific source to all other nodes, APSP considers all possible starting points. A naive approach to APSP is running an SSSP algorithm $n$ times (where $n$ is the number of vertices).

**Running Times for Dense Graphs ($E \approx N^2$):**
* **BFS (Unweighted):** $O(N \cdot (N+E)) \approx O(N^3)$
* **Dijkstra:** $O(N \cdot E \log N) \approx O(N^3 \log N)$
* **Bellman-Ford:** $O(N \cdot (NE)) \approx O(N^4)$

The Floyd-Warshall algorithm provides a solution with a consistent **$O(N^3)$** complexity, which is an improvement over running Dijkstra or Bellman-Ford multiple times on dense graphs.


---

## 2. The Core Logic: Slicing by Intermediate Vertices
The Floyd-Warshall algorithm works in phases. Instead of focusing on the number of edges (like Bellman-Ford), it focuses on the **set of allowed intermediate vertices**.

### The $r$-th Phase Definition
Assume vertices are labeled from $1$ to $n$. In the $r$-th phase, the algorithm calculates the shortest path between any two nodes $u$ and $v$ using only intermediate vertices from the set $\{1, 2, \dots, r\}$.
* **$dist(u, v, r)$**: The cost of the shortest path from $u$ to $v$ where every intermediate vertex is numbered at most $r$.
* When $r = n$, every vertex is a candidate for being an intermediate node, meaning we have found the true shortest path for all pairs.

---

## 3. Recursive Structure and Case Analysis
To compute the values for phase $r$, the algorithm relies on the values already computed in phase $r-1$. For any pair $(u, v)$, there are two possibilities for the shortest path in phase $r$:

### Case 1: The path does NOT use vertex $r$
If the shortest path using vertices up to $r$ does not actually need vertex $r$, then the shortest path is the same as it was in the previous phase.
* **Logic:** $dist(u, v, r) = dist(u, v, r-1)$

### Case 2: The path DOES use vertex $r$
If the path uses vertex $r$ as an intermediate node, the path can be split into two parts:
1.  A path from $u$ to $r$.
2.  A path from $r$ to $v$.
Since we assume no negative cycles (meaning optimal paths are "simple" and don't repeat vertices), these two sub-paths must only use intermediate vertices from the set $\{1, 2, \dots, r-1\}$.
* **Logic:** $dist(u, v, r) = dist(u, r, r-1) + dist(r, v, r-1)$


### The Optimization Step
The algorithm chooses the minimum of these two cases:
```text
dist(u, v, r) = min( dist(u, v, r-1), 
                     dist(u, r, r-1) + dist(r, v, r-1) )
```

---

## 4. Pseudocode Implementation
The algorithm is implemented using three nested loops. The outermost loop iterates through the intermediate vertices ($r$), while the inner loops iterate through all possible pairs of start ($u$) and end ($v$) vertices.

```python
ALGORITHM FloydWarshall(Graph)
    # Initialization (Base Case: r = 0)
    FOR each vertex u from 1 to n:
        FOR each vertex v from 1 to n:
            IF u == v:
                distance[u][v] = 0
            ELSE IF there is an edge from u to v:
                distance[u][v] = weight(u, v)
            ELSE:
                distance[u][v] = INFINITY

    # Main Dynamic Programming Phases
    FOR r from 1 to n: # Intermediate vertex phase
        FOR u from 1 to n: # Source vertex
            FOR v from 1 to n: # Destination vertex
                # Update the shortest path if passing through 'r' is cheaper
                IF distance[u][r] + distance[r][v] < distance[u][v]:
                    distance[u][v] = distance[u][r] + distance[r][v]
    
    RETURN distance matrix
```

---

## 5. Summary of Takeaways
* **APSP vs SSSP:** APSP finds the shortest distance between all possible pairs of nodes, whereas SSSP is relative to a single source.
* **Efficiency:** Floyd-Warshall runs in **$O(N^3)$** time and **$O(N^2)$** space, making it ideal for dense graphs.
* **Intermediate Vertex Strategy:** The algorithm builds solutions incrementally by expanding the set of allowed intermediate vertices one by one.
* **Base Case:** When no intermediate vertices are allowed ($r=0$), the shortest path is simply the weight of the direct edge between nodes.
* **Constraint:** The standard version described assumes non-negative edge weights (to ensure simple paths), though it can be adapted for graphs with negative edges but no negative cycles.